# Day 2 notebook companion

Run in order with synthetic data. Mermaid diagrams render on the website. Setup and shared helpers are embedded; no checkout is required. Learner exercises report NOT ATTEMPTED until implemented. Reference checks are separate. Optional controls also have direct function calls.


In [ ]:
import importlib.metadata
import subprocess
import sys
for package, version in {"cryptography": "50.0.1", "matplotlib": "3.10.6", "ipywidgets": "8.1.7"}.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])
print("Dependencies ready. Restart if an older library was already imported, then run all cells.")


## Shared teaching helpers

Inspect this implementation. TLS uses real SSL objects over memory buffers and temporary test key files; no system trust changes or network listeners. The teaching KDF is not a standardized protocol key schedule.


In [ ]:
"""Day 2 teaching helpers. Real TLS over MemoryBIO; no sockets or trust-store changes."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import ssl
import tempfile
import hashlib
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtendedKeyUsageOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


def make_pki(expired=False):
    """Create an isolated root, intermediate, server, and client for this run."""
    now = datetime.now(timezone.utc)
    keys = {name: ec.generate_private_key(ec.SECP256R1())
            for name in ('root', 'intermediate', 'server', 'client')}
    names = {name: x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Workshop ' + name)])
             for name in keys}
    certs = {}
    for name, issuer, ca, path_length, eku in [
        ('root', 'root', True, 1, None),
        ('intermediate', 'root', True, 0, None),
        ('server', 'intermediate', False, None, ExtendedKeyUsageOID.SERVER_AUTH),
        ('client', 'intermediate', False, None, ExtendedKeyUsageOID.CLIENT_AUTH),
    ]:
        end = now - timedelta(days=1) if expired and name == 'server' else now + timedelta(days=7)
        builder = (x509.CertificateBuilder().subject_name(names[name]).issuer_name(names[issuer])
                   .public_key(keys[name].public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=2)).not_valid_after(end)
                   .add_extension(x509.BasicConstraints(ca=ca, path_length=path_length), critical=True)
                   .add_extension(x509.KeyUsage(digital_signature=True, content_commitment=False,
                       key_encipherment=False, data_encipherment=False, key_agreement=False,
                       key_cert_sign=ca, crl_sign=ca, encipher_only=False, decipher_only=False), critical=True)
                   .add_extension(x509.SubjectKeyIdentifier.from_public_key(keys[name].public_key()), False)
                   .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(keys[issuer].public_key()), False))
        if eku:
            builder = builder.add_extension(x509.ExtendedKeyUsage([eku]), False)
            builder = builder.add_extension(x509.SubjectAlternativeName([
                x509.DNSName('invoice.test' if name == 'server' else 'client.test')]), False)
        certs[name] = builder.sign(keys[issuer], hashes.SHA256())
    return keys, certs


def tls_trial(hostname='invoice.test', trust_root=True, expired=False,
              include_intermediate=True, mtls=False, send_client=True,
              client_wrong_eku=False):
    """Handshake and exchange application bytes. Failures raise ssl.SSLError.

    Private PEM files are disposable teaching keys in a temporary directory.
    Does not implement online revocation, networking, or authorization policy.
    """
    keys, certs = make_pki(expired)
    pem = lambda c: c.public_bytes(serialization.Encoding.PEM)
    with tempfile.TemporaryDirectory(prefix='workshop-pki-') as directory:
        base = Path(directory)
        for name in ('server', 'client'):
            selected = 'server' if name == 'client' and client_wrong_eku else name
            chain = pem(certs[selected])
            if include_intermediate or name == 'client':
                chain += pem(certs['intermediate'])
            (base / (name + '.pem')).write_bytes(chain)
            (base / (name + '.key')).write_bytes(keys[selected].private_bytes(
                serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                serialization.NoEncryption()))
        server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
        client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        for context in (server_context, client_context):
            context.minimum_version = context.maximum_version = ssl.TLSVersion.TLSv1_3
        server_context.load_cert_chain(str(base / 'server.pem'), str(base / 'server.key'))
        if trust_root:
            client_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if mtls:
            server_context.verify_mode = ssl.CERT_REQUIRED
            server_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if send_client:
            client_context.load_cert_chain(str(base / 'client.pem'), str(base / 'client.key'))
        ci, co, si, so = (ssl.MemoryBIO() for _ in range(4))
        client = client_context.wrap_bio(ci, co, server_hostname=hostname)
        server = server_context.wrap_bio(si, so, server_side=True)
        completed = [False, False]

        def transfer():
            for outgoing, incoming in ((co, si), (so, ci)):
                if outgoing.pending:
                    incoming.write(outgoing.read())

        for _ in range(100):
            for index, peer in enumerate((client, server)):
                if not completed[index]:
                    try:
                        peer.do_handshake()
                        completed[index] = True
                    except ssl.SSLWantReadError:
                        pass
            transfer()
            if all(completed):
                break
        else:
            raise RuntimeError('TLS handshake stalled')
        payload = b'synthetic confidential invoice'
        client.write(payload)
        transfer()
        assert server.read(4096) == payload
        return {'version': client.version(), 'cipher': client.cipher()[0],
                'client_authenticated': bool(server.getpeercert()),
                'application_bytes': len(payload)}


def expect_rejection(operation, exceptions):
    """Assert the negative case, without accepting a silent failure."""
    try:
        operation()
    except exceptions:
        return
    raise AssertionError('Expected rejection did not occur')


def derive_day2(secret, transcript, direction=b'alice-to-bob'):
    """Teaching KDF only, not a standardized TLS or hybrid key schedule."""
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                info=b'workshop-day2:v1|' + hashlib.sha256(transcript).digest()
                + b'|' + direction).derive(secret)


# Lab 5: Build an ML-KEM Secure Channel

**60 minutes guided · 90–120 minutes independently.** Notebook (see course website) · Instructor (see course website) · Solutions (see course website)

## Goal and boundary

Replace the classical secret-establishment step with ML-KEM-768, then derive directional keys and protect a record with AES-GCM. Compare its public exchange sizes with X25519. Complete Sessions 4 and 9 and Day 2 setup (see course website). This notebook is self-contained and does not depend on the unfinished Day 1 Lab 2.

The title describes a teaching channel construction. Peer identity is assumed through a provisioned public key; we do not implement a complete authenticated transport, TLS profile, key confirmation, record sequencing or replay protection.



```mermaid
flowchart LR
    X["Classical X25519 baseline"] --> S["Shared material"]
    K["Replace with ML-KEM"] --> S
    S --> H["HKDF with transcript and direction"]
    H --> A["AEAD record"]
```

Read the diagram as two alternative establishment experiments, not the hybrid combiner from Session 10.

## Task one: establish and measure — 15 minutes


In [ ]:
from cryptography.hazmat.primitives.asymmetric import x25519, mlkem
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag
import secrets
a, b = x25519.X25519PrivateKey.generate(), x25519.X25519PrivateKey.generate()
assert a.exchange(b.public_key()) == b.exchange(a.public_key())
bob = mlkem.MLKEM768PrivateKey.generate()
pk = bob.public_key().public_bytes_raw()
sender_secret, ct = bob.public_key().encapsulate()
receiver_secret = bob.decapsulate(ct)
assert sender_secret == receiver_secret
transcript = b'lab5:v1:ML-KEM-768|' + pk + ct
print('X25519 two public contributions:', 32 + 32)
print('ML-KEM-768 public key plus ciphertext:', len(pk) + len(ct))
print('PASS: classical baseline and real KEM agree on their respective secrets')


These are raw contribution lengths, not full handshake sizes or an equivalent-security benchmark. Add credentials, framing and retransmission costs in a deployment measurement.

## Task two: implement derivation — 20 minutes

Implement `learner_key(secret, transcript, direction)` to match the lesson's `derive_day2` teaching schedule. Return 32 bytes from HKDF-SHA-256, with `salt=None` and info consisting of `b'workshop-day2:v1|'`, SHA-256 of the transcript, `b'|'`, and the direction. Using the helper is allowed after you explain each field.


In [ ]:
def learner_key(secret, transcript, direction):
    raise NotImplementedError('Derive a context-bound directional traffic key')

def check_key(candidate):
    ab = candidate(sender_secret, transcript, b'alice-to-bob')
    assert ab == derive_day2(sender_secret, transcript, b'alice-to-bob')
    assert ab == candidate(receiver_secret, transcript, b'alice-to-bob')
    assert ab != candidate(sender_secret, transcript, b'bob-to-alice')
    assert ab != candidate(sender_secret, transcript + b'changed', b'alice-to-bob')
    nonce = secrets.token_bytes(12)
    record = AESGCM(ab).encrypt(nonce, b'synthetic invoice', b'tenant=acme')
    assert AESGCM(ab).decrypt(nonce, record, b'tenant=acme') == b'synthetic invoice'
    expect_rejection(lambda: AESGCM(ab).decrypt(nonce, record, b'tenant=other'), InvalidTag)

try:
    check_key(learner_key)
except NotImplementedError:
    print('NOT ATTEMPTED: learner KDF')
else:
    print('PASS: learner KDF')


<details><summary>Hints</summary><p>Use a new HKDF object for each call. Direction is the traffic direction, not the current caller's identity. Both peers need identical transcript bytes. Omitting the transcript or using the raw KEM secret should fail the checks.</p></details>

## Task three: predict rejection — 15 minutes


In [ ]:
def reference_key(secret, transcript, direction):
    return derive_day2(secret, transcript, direction)
check_key(reference_key)
print('PASS: supplied Lab 5 reference checks')

key = reference_key(sender_secret, transcript, b'alice-to-bob')
nonce = secrets.token_bytes(12)
aad = b'invoice=7|tenant=acme'
record = AESGCM(key).encrypt(nonce, b'lab5 confidential message', aad)
def observe_record(case='valid'):
    candidate_secret, candidate_aad, candidate_record = receiver_secret, aad, record
    if case == 'changed KEM ciphertext':
        altered = bytes([ct[0] ^ 1]) + ct[1:]
        candidate_secret = bob.decapsulate(altered)
    elif case == 'wrong recipient':
        candidate_secret = mlkem.MLKEM768PrivateKey.generate().decapsulate(ct)
    elif case == 'changed AAD':
        candidate_aad += b'!'
    elif case == 'changed record':
        candidate_record = bytes([record[0] ^ 1]) + record[1:]
    candidate_key = reference_key(candidate_secret, transcript, b'alice-to-bob')
    try:
        AESGCM(candidate_key).decrypt(nonce, candidate_record, candidate_aad)
    except InvalidTag:
        return 'REJECTED'
    return 'ACCEPTED'
assert observe_record() == 'ACCEPTED'
for case in ('changed KEM ciphertext', 'wrong recipient', 'changed AAD', 'changed record'):
    assert observe_record(case) == 'REJECTED'
expect_rejection(lambda: bob.decapsulate(ct[:-1]), ValueError)
print('PASS: KEM/record failures rejected at the appropriate boundary')


In [ ]:
if 'get_ipython' in globals():
    import ipywidgets as widgets
    from IPython.display import display
    display(widgets.interactive(observe_record, case=['valid', 'changed KEM ciphertext',
        'wrong recipient', 'changed AAD', 'changed record']))


The direct function call is the widget fallback. Same-length invalid KEM ciphertext can yield a fallback secret without an exception; AEAD then fails. Do not treat returned secret length as peer authentication.

## Debrief and submission — 10 minutes

Submit the learner KDF, measured byte counts, failure observations, and three missing protocol protections. Explain why replaying the unchanged record still passes this example and why retaining Bob's decapsulation key can expose recorded exchanges after compromise. State how Bob's public key would become trusted; merely sending it alongside the ciphertext does not solve identity.

Extension: benchmark repeated operations using monotonic timing and report distributions plus platform/version. Keep such measurements separate from algorithm security claims. See solutions (see course website).


In [ ]:
print("PASS: completed lab-05-pqc-channel demonstrations; learner status is reported separately")
